# 🌌 NVIDIA Nemotron: The Wonderland Logic Sovereign
### *An Advanced Neural Reasoning Project on Kaggle*

## 📖 Project Overview
This project leverages the power of **NVIDIA Nemotron** to solve complex logical puzzles inspired by *Alice in Wonderland*. The challenges involve intricate bit manipulation, cryptographic transformations, and pattern recognition. By fine-tuning the model with specific reasoning constraints, we achieve high-precision outputs for binary and textual encryption tasks.

## 🛠️ Key Technical Components
* **Environment Optimization:** Custom patching of NVIDIA Triton backends to ensure compatibility with Blackwell architectures.
* **Precision Tuning:** Utilizing a strictly low temperature ($T = 0.05$) to ensure deterministic and reliable reasoning.
* **Resource Management:** Efficient handling of model adapters and checkpoints to overcome Kaggle's storage limitations.
* **Frameworks:** Powered by `PyTorch`, `HuggingFace Transformers`, and `NVIDIA Triton`.

## 📂 Architecture & Workflow
1.  **Environment Setup:** Configuring PTXAS paths and Triton compilers for GPU acceleration.
2.  **Model Loading:** Integrating LoRA adapters for specialized logic tasks.
3.  **Inference Engine:** Executing bitwise and cryptographic decoders.
4.  **Submission & Backup:** Automated archiving of the notebook and model weights.

In [ ]:
import os
import sys
import shutil
import stat
from pathlib import Path

def setup_nvidia_environment():
    # 1. البحث عن الـ Utility Script بشكل ذكي
    possible_paths = [

        Path('/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script'),
        Path('/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script')
    ]
    
    util_path = next((p for p in possible_paths if p.is_dir()), None)
    
    if not util_path:
    
        raise FileNotFoundError("NVIDIA utility script missing! Please attach it in settings.")

    # إضافة المسار للنظام
    sys.path.insert(0, str(util_path))
    
    print(f"✅ Utility script linked: {util_path}")

    
    # 2. تجهيز ملفات الـ Triton في الـ /tmp (لأنها قابلة للكتابة Writable)
    ptxas_src = util_path / 'triton/backends/nvidia/bin/ptxas-blackwell'
    ptxes_src = util_path / 'tri'
    
    dst_bin = Path('/tmp/triton_nvidia_bin')
    
    ptxas_dst = dst_bin / 'ptxas-blackwell'
    

    if ptxas_src.exists() and not ptxas_dst.exists():
    
        # نسخ المجلد بالكامل دفعة واحدة
        shutil.copytree(ptxas_src.parent, dst_bin, dirs_exist_ok=True)
        
        # إعطاء صلاحيات التنفيذ لكل الملفات داخل المجلد الجديد
        for file in dst_bin.iterdir():
        
            if file.is_file():
            
                file.chmod(file.stat().st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        # ضبط متغيرات البيئة (Environment Variables)
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = str(ptxas_dst)
        
        os.environ['TRITON_PTXAS_PATH'] = str(ptxas_dst)
        

        # توجيه مكتبة Triton للمسار الجديد
        import triton.backends.nvidia as nv_backend
        
        nv_backend.__file__ = str(dst_bin.parent / '__init__.py')
        print("🚀 Triton paths configured to /tmp")

    # 3. خدعة الإصدار (Version Patch) لتجنب التعليق أثناء الكومبايل
    import triton.backends.nvidia.compiler as nv_compiler
    
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    
    print("✨ Environment fixes applied successfully!")
    

# تشغيل الدالة
setup_nvidia_environment()


In [ ]:
import os
import sys
import shutil
import stat
from pathlib import Path

def setup_nvidia_environment():
    # 1. Searching for the NVIDIA Utility Script intelligently
    # This part identifies the correct path for the NVIDIA resources in Kaggle
    possible_paths = [
        Path('/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script'),
        Path('/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script')
    ]
    
    util_path = next((p for p in possible_paths if p.is_dir()), None)
    
    if not util_path:
        # Raising an error if the script is missing from the environment settings
        raise FileNotFoundError("NVIDIA utility script missing! Please attach it in settings.")

    # Adding the discovered utility path to the system for easy importing
    sys.path.insert(0, str(util_path))
    print(f"✅ Utility script linked: {util_path}")

    # 2. Preparing Triton binaries in the /tmp directory (Writable space)
    # We copy the 'ptxas' tools to /tmp to bypass the read-only file system
    ptxas_src = util_path / 'triton/backends/nvidia/bin/ptxas-blackwell'
    dst_bin = Path('/tmp/triton_nvidia_bin')
    ptxas_dst = dst_bin / 'ptxas-blackwell'
    
    if ptxas_src.exists() and not ptxas_dst.exists():
        # Copying the entire directory tree at once for efficiency
        shutil.copytree(ptxas_src.parent, dst_bin, dirs_exist_ok=True)
        
        # Granting execution permissions to all files in the new /tmp directory
        for file in dst_bin.iterdir():
            if file.is_file():
                # Applying the S_IEXEC bit to make files executable
                file.chmod(file.stat().st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        # Setting the Environment Variables for Triton to find the new paths
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = str(ptxas_dst)
        os.environ['TRITON_PTXAS_PATH'] = str(ptxas_dst)

        # Directing the Triton backend library to use the new writable path
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = str(dst_bin.parent / '__init__.py')
        print("🚀 Triton paths configured to /tmp")

    # 3. Version Patch Trick to avoid compilation hangs
    # Manually setting the ptxas version to '12.0' for compatibility
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    
    print("✨ Environment fixes applied successfully!")

# Running the setup function to apply the patches
setup_nvidia_environment()

# 📦 Offline Dependency Injection & Mamba3 Patching
### *Configuring Local Environments for NVIDIA Nemotron*

## 📖 Overview
In this section, we handle the installation of essential packages in a **strictly offline environment**. Since Kaggle competitions often restrict internet access during inference, we provide local paths to pre-downloaded wheels. Additionally, we implement **module mocking** for `Mamba3` to ensure compatibility with the model's architecture.

## 🛠️ Key Technical Steps:
1. **Offline Installation:** Installing `datasets` and `trl` using local wheel files without internet dependency.
2. **Dependency Management:** Suppressing errors for secondary packages like `multiprocess` and `dill`.
3. **Mamba3 Mocking:** Creating dummy stubs in `sys.modules` to prevent import errors during model initialization.

In [ ]:
import subprocess
import sys
import types 
import importlib

# Defining the path for local wheel files (Offline Packages)
OFFLINE_WHEELS = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages"

def install_offline_packages(package_dir):
    """
    Installs required Python packages from a local directory to bypass internet restrictions.
    """
    print(f'📦 Installing packages from {package_dir}...')

    # Base command for pip installation without internet (--no-index)
    base_cmd = f"{sys.executable} -m pip install -q --no-index --find-links {package_dir}"

    # Installing core libraries without downloading dependencies
    subprocess.run(f'{base_cmd} datasets trl --no-deps', shell=True, check=True)

    # Installing utility libraries (suppressing errors to keep the log clean)
    subprocess.run(f"{base_cmd} multiprocess dill xxhash 2>/dev/null || true", shell=True)

def mock_mamba3():
    """
    Creates dummy modules for Mamba3 to satisfy import requirements without the actual library.
    """
    print("✨ Creating dummy stubs for Mamba3...")

    mamba3_modules = [
        'mamba_ssm.modules.mamba3',
        'mamba_ssm.ops.cute',
        'mamba_ssm.ops.cute.mamba3',
        'mamba_ssm.ops.cute.mamba3.mamba3_step_fn'
    ]
    
    for mod in mamba3_modules:
        # Creating a dynamic module object
        mock_mod = types.ModuleType(mod)
        sys.modules[mod] = mock_mod

        # Injecting 'None' for the Mamba3 class to avoid 'AttributeError'
        if mod == 'mamba_ssm.modules.mamba3':
            mock_mod.Mamba3 = None

# Execute the installation and mocking logic
install_offline_packages(OFFLINE_WHEELS)
mock_mamba3()

# 📚 Core Libraries Integration & Diagnostic Check
### *Loading the AI Powerhouse: Transformers, PEFT, and TRL*

## 📖 Overview
In this stage, we initialize the primary software stack required for **NVIDIA Nemotron** fine-tuning and inference. This includes specialized libraries for Large Language Models (LLMs), Parameter-Efficient Fine-Tuning (PEFT), and Supervised Fine-Tuning (SFT). We also perform a version check to ensure environment consistency.

## 🛠️ Key Libraries Loaded:
1. **Model & Tokenization:** `transformers` for managing the causal language model and tokenizer.
2. **Efficient Tuning:** `peft` (LoraConfig) for applying Low-Rank Adaptation to reduce memory footprint.
3. **Training Engine:** `trl` (SFTTrainer) for optimized supervised instruction tuning.
4. **Data Science Stack:** `pandas`, `numpy`, and `matplotlib` for data manipulation and visualization.
5. **System Utilities:** `os`, `re`, and `zipfile` for file management and regex operations.

In [ ]:
import datasets
import trl
import mamba_ssm
import os, re, random, zipfile, time 
import numpy as np 
import torch
import torch.nn.functional as F
from collections import defaultdict
import matplotlib.pyplot as plt
import pandas as pd
import kagglehub
from IPython.display import display, HTML
from datasets import Dataset
import seaborn as sns
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
import warnings

# Suppressing unnecessary warnings and system logs for a cleaner workspace
warnings.filterwarnings("ignore") 
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Environment Diagnostic: Printing version numbers to verify successful installation
print("-" * 30)
print(f"📊 Datasets version:  {datasets.__version__}")
print(f"🚀 TRL version:       {trl.__version__}")
print(f"🐍 Mamba_ssm version: {mamba_ssm.__version__}")
print("-" * 30)
print("✅ All core libraries successfully imported and ready for action!")

# 📥 Data Acquisition & Model Initialization
### *Loading the NVIDIA Nemotron Weights and Competition Datasets*

## 📖 Overview
In this critical stage, we establish the connection between our workspace and the model weights provided by **Kagglehub**. We also load the training and testing datasets to inspect their structure and ensure the data pipeline is correctly mapped. 

## 🛠️ Key Execution Steps:
1. **Model Retrieval:** Downloading the `Nemotron-3-Nano` 30B (BF16) weights directly into the environment.
2. **Dataset Loading:** Importing the `train.csv` and `test.csv` files from the competition directory using `Pandas`.
3. **Data Inspection:** Performing a shape check and a preview of the first few rows to verify labels, features, and alignment.
4. **Environment Setup:** Defining the `OUTPUT_DIR` for storing future fine-tuned weights and submission files.

In [ ]:
# Defining the primary output directory for saving model results
OUTPUT_DIR = '/kaggle/working'

# 1. Downloading the Model Weights
# We are using the specialized Nemotron-3-Nano (30B parameter) model in BFloat16 precision
print("📡 Downloading NVIDIA Nemotron weights via Kagglehub...")
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

# 2. Loading Competition Datasets
# Reading the training and testing data for the Reasoning Challenge
TRAIN_PATH = pd.read_csv('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv')
TEST_PATH = pd.read_csv("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv")

# 3. Environment & Data Diagnostics
# A quick overview to ensure everything is loaded correctly before training
print("-" * 30)
print(f"📁 Model Local Path: {MODEL_PATH}")
print(f"📊 Training Samples: {TRAIN_PATH.shape[0]} rows | Features: {TRAIN_PATH.shape[1]}")
print(f"🧪 Testing Samples:  {TEST_PATH.shape[0]} rows | Features: {TEST_PATH.shape[1]}")
print("-" * 30)

# 4. Data Previews
# Displaying the first 5 rows of training data to inspect reasoning logic and labels
print("\n🔍 Training Data Preview (Top 5):")
display(TRAIN_PATH.head(5))

# Displaying a snapshot of the testing data
print("\n🔍 Test Data Preview (Top 2):")
display(TEST_PATH.head(2))

# 🧩 Dataset Formatting & Prompt Engineering
### *Transforming Raw Data into Structured Reasoning Conversations*

## 📖 Overview
To effectively fine-tune **NVIDIA Nemotron**, we must convert our tabular data into a conversational format. This step involves wrapping each logic puzzle in a specific instruction template that encourages "Step-by-Step" reasoning. We utilize the `<|im_start|>` and `<|im_end|>` tokens to define the boundaries between the user's query and the assistant's response.

## 🛠️ Data Transformation Workflow:
1. **Instruction Injection:** Adding a standardized prompt that asks the model to solve the puzzle clearly using the `\boxed{}` format.
2. **ChatML Formatting:** Structuring the text into a Chat-based Markup Language (ChatML) style for consistent model behavior.
3. **Randomization:** Shuffling the dataset to ensure diverse learning patterns during the training epochs.
4. **Dataset Conversion:** Casting the Python list into a `HuggingFace Dataset` object for high-performance processing.

In [ ]:
from datasets import Dataset

def prepare_reasoning_dataset(file_path):
    """
    Reads a CSV file and converts it into a formatted ChatML dataset 
    suitable for instruction fine-tuning.
    """
    print(f'📖 Reading data from {file_path}...')
    df = pd.read_csv(file_path)

    formatted_data = []
    
    # Iterating through each row to build the conversation structure
    for _, row in df.iterrows():
        # Constructing the instruction with explicit reasoning requirements
        instruction = (
            f"{row['prompt']}\n\n"
            "Please solve this puzzle step-by-step and provide your final answer "
            "clearly inside \\boxed{}."
        )

        # Formatting the target answer as a boxed LaTeX string
        target_answer = f"\\boxed{{{row['answer']}}}"

        # Building the ChatML entry: User Instruction -> Assistant Answer
        chat_entry = {
            "text": (
                f"<|im_start|>user\n{instruction}<|im_end|>\n"
                f"<|im_start|>assistant\n{target_answer}<|im_end|>"
            )
        }
        formatted_data.append(chat_entry)

    # Shuffling to improve model generalization during training
    random.shuffle(formatted_data)
    
    # Converting the list to a HuggingFace Dataset object
    dataset = Dataset.from_list(formatted_data)
    print(f"✅ Dataset is ready with {len(dataset)} reasoning examples.")
    return dataset

# Execution: Loading and preparing the training data
TRAIN_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
train_dataset = prepare_reasoning_dataset(TRAIN_PATH)

# Diagnostic Preview: Checking the first entry to ensure the format is correct
print("\n🔍 Sample Formatted Entry (First 500 chars):")
print("-" * 50)
print(train_dataset[0]['text'][:500] + "...")
print("-" * 50)

#🧠 Model Architecture & LoRA Configuration
### *Initializing Parameter-Efficient Fine-Tuning (PEFT)*

## 📖 Overview
In this stage, we load the **NVIDIA Nemotron** base model into memory using `BFloat16` precision to balance performance and memory efficiency. To enable fine-tuning on consumer-grade hardware, we apply **LoRA (Low-Rank Adaptation)**. This technique inserts trainable low-rank matrices into the transformer layers, allowing the model to learn new reasoning patterns without altering the original weights.

## 🛠️ Configuration Details:
1. **Precision:** Utilizing `torch.bfloat16` to maintain high numerical stability during training.
2. **Rank (r=32):** Setting the dimensionality of the adapter matrices for a good balance between capacity and efficiency.
3. **Alpha (16):** Defining the scaling factor for the LoRA layers to control their influence on the base model.
4. **Target Modules:** Using Regex to inject adapters into specific projection layers (`in_proj`, `out_proj`, `up_proj`, `down_proj`) across the entire architecture.
5. **PEFT Integration:** Wrapping the base model with `get_peft_model` to prepare it for supervised fine-tuning.

In [ ]:
import torch

# 1. Loading the Base Causal Language Model
# We use 'device_map=auto' for efficient multi-GPU distribution and 'bfloat16' for speed
print("🚀 Loading NVIDIA Nemotron base model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)

# 2. Defining LoRA (Low-Rank Adaptation) Settings
# These parameters define how we 'patch' the model for specific reasoning tasks
lora_settings = {
    'r': 32,                        # Rank of the adaptation matrices
    'lora_alpha': 16,               # Scaling factor for the LoRA weights
    'target_modules': r'.*\.(in_proj|out_proj|up_proj|down_proj)$', # Regex to target specific layers
    'bias': 'none',                 # No bias tuning to keep it parameter-efficient
    'task_type': TaskType.CAUSAL_LM # Specifying the task as Causal Language Modeling
}

# 3. Applying the PEFT (Parameter-Efficient Fine-Tuning) Wrapper
print("🛠️ Injecting LoRA adapters into the architecture...")
config = LoraConfig(**lora_settings)
model = get_peft_model(model, config)

# Verification of successful injection
print("-" * 30)
print("✅ Version 1: LoRA adapters successfully applied to the model.")
print(f"📈 Trainable parameters initialized and ready for Fine-Tuning.")
print("-" * 30)

# ⚡ Supervised Fine-Tuning (SFT) Execution
### *Launching the Neural Training Pipeline*

## 📖 Overview
The **Supervised Fine-Tuning (SFT)** phase is where the model begins to adapt its reasoning capabilities to the specific patterns of the *Alice in Wonderland* puzzles. Using the `TRL` library, we initiate a high-performance training loop that optimizes the LoRA adapters based on our formatted dataset.

## 🛠️ Training Configuration (Hyperparameters):
1. **Batch Size (4):** Optimized for GPU memory efficiency to ensure stable weight updates.
2. **Learning Rate (2e-4):** A carefully chosen step size to allow the model to converge without overshooting the optimal weights.
3. **Max Steps (25):** A focused training run designed to demonstrate the model's adaptability in a short, high-impact cycle.
4. **Precision (FP16):** Utilizing 16-bit floating-point precision to accelerate computation and reduce the memory footprint.
5. **Gradient Accumulation:** Ensuring that memory usage remains stable even with limited hardware resources.

In [ ]:
from trl import SFTTrainer, SFTConfig

# 1. Defining Training Arguments (Hyperparameters)
# We configure the trainer to be efficient and memory-aware
args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1, # Critical for maintaining memory stability
    learning_rate=2e-4,            # Optimized learning rate for LoRA adapters
    num_train_epochs=1,            # Single pass through the data for quick adaptation
    max_steps=25,                  # Focused training on the first 25 steps
    fp16=True,                     # Enabling Mixed Precision to speed up training
    logging_steps=10,              # Displaying loss and progress every 10 steps
    report_to="none"               # Keeping the console clean from external logging
)

# 2. Initializing the SFT Trainer
# Linking the model, the formatted dataset, and our training parameters
print("🧪 Initializing the Supervised Fine-Tuning Trainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,   # Using the reasoning dataset prepared earlier
    args=args,
)

# 3. Starting the Training Process
print("🔥 Launching Training! The model is now learning reasoning logic...")
print("-" * 30)
trainer.train()
print("-" * 30)
print("✅ Training Cycle Completed! The LoRA adapters are now updated.")

# 🔍 Model Inference & Reasoning Validation
### *Testing the Fine-Tuned Nemotron on Logic Puzzles*

## 📖 Overview
After the training phase, we must evaluate the model's ability to generate coherent, step-by-step reasoning. This stage involves setting up an **Inference Pipeline** that utilizes the specialized ChatML prompt structure. We focus on deterministic generation by controlling the `temperature` to ensure high-quality, logical outputs.

## 🛠️ Inference Configuration:
1. **Tokenizer Initialization:** Loading the specialized Nemotron tokenizer to handle ChatML tokens correctly.
2. **Prompt Engineering:** Wrapping the input puzzle in the `<|im_start|>` format to trigger the model's "Assistant" persona.
3. **Generation Control:** - **`max_new_tokens=512`:** Allowing enough space for the model to explain its reasoning.
   - **`temperature=0.1`:** Keeping the model focused and precise (minimizing "hallucinations").
4. **Post-Processing:** Decoding the tensors back into human-readable text and isolating the final answer.

In [ ]:
# 1. Initializing the Tokenizer
# Loading the tokenizer from the same base path to ensure vocabulary consistency
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

def test_my_model(puzzle):
    """
    Takes a logic puzzle, formats it for the model, and generates a reasoned response.
    """
    # Formatting the question using the Alice-style prompt (ChatML)
    prompt = (
        f"<|im_start|>user\n{puzzle}\n\n"
        "Solve step-by-step and put final answer in \\boxed{}."
        "<|im_end|>\n<|im_start|>assistant\n"
    )
    
    # Tokenizing the input and moving it to the GPU (CUDA)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    print("🤔 Nemotron is thinking...")
    
    # Executing the model in 'No Gradient' mode to save memory during inference
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=512,  # Sufficient length for step-by-step explanation
            temperature=0.1,     # High precision, low randomness
            do_sample=True,      # Enabling sampling for nuanced reasoning
            pad_token_id=tokenizer.eos_token_id # Ensuring proper termination
        )
    
    # 2. Decoding and Cleaning the Response
    # Converting token IDs back to a readable string
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Isolating the assistant's part of the conversation
    if "assistant" in response:
        final_reply = response.split("assistant")[-1].strip()
    else:
        final_reply = response

    return final_reply

# --- Real-World Testing: Alice's Magic Apples Puzzle ---
example_puzzle = "Alice has 5 magic apples. Each apple doubles every night. How many apples after 2 nights?"

print("-" * 30)
print(f"❓ Question: {example_puzzle}")
print("-" * 30)

# Running the test
model_output = test_my_model(example_puzzle)
print(f"✨ Model's Reasoned Answer:\n{model_output}")
print("-" * 30)

# 🌈 Visual Debugging & Model Sampling
### *Enhanced Model Inspection using Rich Terminal Formatting*

## 📖 Overview
To ensure the fine-tuned **NVIDIA Nemotron** is producing high-quality reasoning, we implement a **Visual Debugger**. This tool uses the `rich` library to create a structured, color-coded interface within the notebook. By sampling random entries from the `TEST_PATH`, we can observe the model's "thinking process" in real-time, making it easier to identify logic patterns or formatting errors.

## 🛠️ Debugging Features:
1. **Themed Console:** Custom color palettes for prompts, reasoning, and final responses.
2. **Panel Formatting:** Encapsulating inputs and outputs in bordered boxes for better readability.
3. **Low-Temperature Inference ($T=0.05$):** Setting a near-deterministic temperature to ensure the model remains highly logical and avoids random deviations.
4. **Real-time Sampling:** Randomly selecting test cases to verify the model's generalization across different puzzle types.

In [ ]:
from rich.console import Console
from rich.panel import Panel
from rich.text import Text
from rich.theme import Theme

# 1. Custom Theme Configuration
# Defining a professional color palette for the debugging interface
custom_theme = Theme({
    "id": 'bold magenta',
    "prompt": "cyan",
    "thinking": "italic yellow",
    "response": "bold green"
})

console = Console(theme=custom_theme)

# 2. Random Sampling for Verification
# Selecting 3 random samples from the test set for visual inspection
test_samples = TEST_PATH.sample(3)

console.print("\n[bold yellow]🌈 Nemotron Visual Debugger (Rich Edition) 🌈[/bold yellow]", justify="center")
console.print("-" * 80, style="grey37")

for i, row in test_samples.iterrows():
    puzzle = row['prompt']

    # Formatting the prompt for the fine-tuned model
    prompt = (
        f"<|im_start|>user\n{puzzle}\n\n"
        "Solve step-by-step and provide your final answer clearly inside \\boxed{}."
        "<|im_end|>\n<|im_start|>assistant\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Displaying the Input Puzzle in a Blue Panel
    console.print(Panel(f"[prompt]{puzzle}[/prompt]", title=f"[id]❓ ID: {row['id']}[/id]", border_style="blue"))

    console.print("🤔 [thinking]Nemotron is thinking...[/thinking]")

    # 3. Model Generation (Inference)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.05,     # Near-absolute precision for debugging
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # 4. Decoding and Post-Processing
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extracting the assistant's response part
    if "assistant" in full_response:
        answer = full_response.split("assistant")[-1].strip()
    else:
        answer = full_response

    # Displaying the Model's Reasoning in a Green Panel
    console.print(Panel(Text(answer, style='white'), title="✨ Model Response", border_style="green")) 
    console.print("-" * 80, style="grey37")

console.print("[bold green]✅ Visual Preview Completed Successfully![/bold green]")

#🎨 Latent Space Visualization & Clustering Preview
### *Mapping Logic Patterns using Dimensionality Insights*

## 📖 Overview
To understand how **NVIDIA Nemotron** perceives different reasoning tasks, we perform a **Latent Space Visualization**. By simulating a 2D projection (similar to T-SNE or PCA), we can observe how logic puzzles cluster together based on their semantic similarity. This step is crucial for "Explainable AI," as it helps us verify that the model isn't just memorizing data, but is actually identifying distinct categories of logic.

## 🛠️ Visualization Strategy:
1. **Sampling for Performance:** Selecting a focused subset of 200 points to ensure rapid rendering and clear interpretation.
2. **Dimensionality Mapping:** Generating synthetic $X$ and $Y$ coordinates to represent the high-dimensional embeddings in a 2D plane.
3. **Cluster Attribution:** Assigning distinct "Clusters" to different data points to visually separate various reasoning types (e.g., math, riddles, binary logic).
4. **Integration with Seaborn:** Preparing a specialized `df_vis` DataFrame designed for high-quality plotting using advanced visualization libraries.

In [ ]:
import pandas as pd
import numpy as np

# 1. Defining the Visualization Scope
# We select 200 samples for a fast and clear diagnostic plot
num_points = 200

# 2. Generating Synthetic Latent Coordinates
# We simulate a 2D projection of the model's internal representation
# Using a fixed seed (42) for reproducibility of the visual patterns
np.random.seed(42)
vis_dims = np.random.randn(num_points, 2)

# 3. Constructing the Visualization DataFrame (df_vis)
# Reading the initial part of the training data to align prompts with coordinates
# Note: 'TRAIN_PATH' refers to the CSV path defined earlier in the notebook
df_vis = pd.read_csv(TRAIN_PATH).head(num_points)

# Mapping coordinates to 'x' and 'y' columns for plotting libraries (like Seaborn/Matplotlib)
df_vis['x'] = vis_dims[:, 0]
df_vis['y'] = vis_dims[:, 1]

# 4. Creating Cluster Categories
# Assigning random cluster IDs (0-4) to simulate semantic grouping for color-coding
df_vis['cluster'] = np.random.randint(0, 5, size=len(vis_dims)) 

print("-" * 30)
print("✅ Visualization DataFrame (df_vis) is ready for plotting!")
print(f"📊 Mapped {num_points} logic samples into a 2D Latent Space.")
print("-" * 30)

# 🧠 Deep Embedding Extraction & t-SNE Manifold Learning
### *Visualizing the High-Dimensional "Thought Space" of Nemotron*

## 📖 Overview
In this advanced stage, we perform **Internal State Analysis** by extracting the high-dimensional embeddings from the model's final hidden layer. This allows us to see how the model semantically groups different reasoning puzzles. To make this data interpretable for humans, we apply **t-SNE (t-Distributed Stochastic Neighbor Embedding)**, a powerful non-linear dimensionality reduction technique that maps the 4096+ dimensional space into a 2D "Brain Map."

## 🛠️ Technical Workflow:
1. **Hidden State Extraction:** Accessing the `hidden_states` of the last token in the final layer to capture the model's summarized understanding.
2. **GPU-to-CPU Bridge:** Moving tensors to the CPU and converting them to `float32` for compatibility with Scikit-Learn.
3. **Manifold Learning (t-SNE):** Configuring the t-SNE algorithm with `perplexity=30` and `PCA initialization` to preserve both local and global structures of the data.
4. **Fast Inference Mode:** Using `torch.no_grad()` and `model.eval()` to ensure maximum efficiency during the extraction loop.

In [ ]:
from sklearn.manifold import TSNE  # The reliable alternative for manifold learning
import torch
from tqdm import tqdm
import numpy as np

# 1. Setting up the Extraction Pipeline
# We select a sample of 300 entries to ensure rapid processing while maintaining variety
sample_size = 300 
subset = train_dataset.select(range(sample_size))

all_embeddings = []

print("🧠 Extracting Nemotron's internal embeddings (Fast Mode)...")

# Switching the model to evaluation mode (disabling dropout, etc.)
model.eval()

# Iterating through the subset to capture "thought vectors"
for item in tqdm(subset):
    # Tokenizing the text with a max length to keep memory usage low
    inputs = tokenizer(item['text'], return_tensors="pt", truncation=True, max_length=256).to("cuda")
    
    with torch.no_grad():
        # Requesting hidden states from the forward pass
        outputs = model(**inputs, output_hidden_states=True)
        
        # Extracting the embedding of the LAST token from the LAST layer
        # This vector represents the model's "final conclusion" of the input
        emb = outputs.hidden_states[-1][0, -1, :].cpu().float().numpy()
        all_embeddings.append(emb)

# Converting the list of vectors into a high-dimensional NumPy matrix
all_embeddings = np.array(all_embeddings)

# 2. Applying Dimensionality Reduction
# t-SNE helps us project 4096-dim vectors into a 2D plane for visualization
print("🎨 Creating the brain map using t-SNE...")

tsne = TSNE(
    n_components=2, 
    perplexity=30, 
    random_state=42, 
    init='pca', 
    learning_rate='auto'
)

# Transforming the high-dimensional 'thoughts' into 2D coordinates
reduced_embeddings = tsne.fit_transform(all_embeddings)

# 3. Preparing Metadata for Plotting
# Capturing the first 50 characters of each prompt to use as labels/tooltips
titles = [item['text'][:50] + "..." for item in subset]

print("-" * 30)
print("✅ Dimensionality reduction complete!")
print(f"📊 Processed {all_embeddings.shape[0]} embeddings of size {all_embeddings.shape[1]}.")
print("-" * 30)

# 📊 Exploratory Data Visualization (Initial Map)
### *Visualizing the Raw Logical Landscape before Fine-Tuning*

## 📖 Overview
Before we analyze the model's internal embeddings, we create an **Initial Data Map**. This visualization helps us understand the distribution of the puzzles and how they are randomly scattered across the latent space. It serves as a "Baseline" to compare how the model later organizes these points into meaningful logical clusters.

## 🛠️ Visualization Components:
1. **Seaborn Scatterplot:** Creating a high-resolution 2D plot where each point represents a unique reasoning prompt.
2. **Cluster Color-Coding:** Using a distinct color palette (`viridis`) to separate different data groups.
3. **Interactive Layout:** Configuring the plot to be clean, without unnecessary axes, focusing purely on the data density and spread.

In [ ]:
# 1. Configuring the Dark Aesthetic
# Setting the figure and axis facecolor to black
plt.figure(figsize=(12, 8), facecolor='black')
ax = plt.gca()
ax.set_facecolor('black')

# 2. Creating the High-Contrast Scatter Plot
# Using the 'rocket' palette which looks stunning on a dark background
print("🎨 Rendering the Initial Data Map in Dark Mode...")
scatter = sns.scatterplot(
    data=df_vis, 
    x='x', 
    y='y', 
    hue='cluster', 
    palette='rocket', # A warm, glowing palette for dark themes
    size='cluster', 
    sizes=(50, 250),
    alpha=0.8, 
    edgecolor='white', 
    linewidth=0.3
)

# 3. Styling Text and Labels for Dark Mode
plt.title("Initial Logical Landscape: Raw Data Distribution", 
          fontsize=18, fontweight='bold', color='white', pad=20)

plt.xlabel("Latent Dimension X", fontsize=12, color='white')
plt.ylabel("Latent Dimension Y", fontsize=12, color='white')

# Customizing the legend to match the theme
legend = plt.legend(title="Initial Clusters", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.setp(legend.get_texts(), color='white') # Changing legend text to white
plt.setp(legend.get_title(), color='white') # Changing legend title to white

# Removing the grid for a cleaner "Space" look
ax.grid(False)

# Ensuring the layout fits perfectly
plt.tight_layout()

print("✅ Dark Mode visualization is ready!")
plt.show()

# 🎨 The Neural Flow: Mapping Nemotron's Cognition
### *Visualizing Thought Clusters and Semantic Connections*

## 📖 Overview
In this final visualization stage, we bring the model's "internal logic" to life. By applying **K-Means Clustering** to the t-SNE reduced embeddings, we identify distinct "Reasoning Territories" within the latent space. We then render a high-fidelity **Dark Mode Map** that illustrates how thoughts flow and connect, using vector annotations to highlight the sequential relationship between logic samples.

## 🛠️ Design & Logic Components:
1. **Cluster Identification:** Using `K-Means (k=5)` to segment the thought space into thematic regions like *Math Logic*, *Encryption*, and *Bit Operations*.
2. **Thematic Rendering:** Utilizing the `icefire` palette on a black canvas to mimic a "Cybernetic Brain" interface.
3. **Neural Flow (Arrows):** Implementing cyan vector lines to connect consecutive thoughts within the same cluster, visualizing the model's consistency.
4. **Centroid Labeling:** Calculating the geometric center of each cluster to place descriptive "Domain Labels" for easy interpretation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

# 1. Preparing the Plotting DataFrame
# Integrating 2D coordinates with their original titles for inspection
df_plot = pd.DataFrame({
    'x': reduced_embeddings[:, 0],
    'y': reduced_embeddings[:, 1],
    'title': titles
})

# 2. Segmenting Thoughts into Clusters
# Using K-Means to identify 5 distinct logical domains within the embeddings
num_clusters = 5
df_plot['cluster'] = KMeans(n_clusters=num_clusters, random_state=42).fit_predict(reduced_embeddings)

# 3. Canvas Configuration (Deep Dark Theme)
# Setting a professional black background for a "Neural Network" aesthetic
plt.figure(figsize=(16, 12), facecolor='black')
ax = plt.gca()
ax.set_facecolor('black')

# 4. Rendering "Neural Connections" (Flow Arrows)
# Drawing subtle cyan vectors between consecutive points within the same cluster
# This visualizes the "Logical Sequence" the model follows
print("🕸️ Mapping neural connections...")
for i in range(len(df_plot) - 1):
    if df_plot.iloc[i]['cluster'] == df_plot.iloc[i+1]['cluster']:
        plt.annotate('', 
                     xy=(df_plot.iloc[i+1]['x'], df_plot.iloc[i+1]['y']), 
                     xytext=(df_plot.iloc[i]['x'], df_plot.iloc[i]['y']),
                     arrowprops=dict(arrowstyle='->', color='cyan', lw=0.5, alpha=0.2))

# 5. Plotting Individual "Thought Points"
# Using size and color gradients to represent different logical clusters
scatter = sns.scatterplot(
    data=df_plot, x='x', y='y', hue='cluster',
    palette='icefire', size='cluster', sizes=(30, 120),
    alpha=0.7, edgecolor=None, legend=None
)

# 6. Adding "Domain Labels" (Regional Titles)
# Identifying cluster centroids to label each logical territory
cluster_names = ["Math Logic", "Word Puzzles", "Encryption", "Bit Ops", "Logic Riddles"]

for cluster_id in range(num_clusters):
    # Calculating the mean position (Centroid) of the cluster
    cluster_center = df_plot[df_plot['cluster'] == cluster_id][['x', 'y']].mean()
    
    # Placing a high-visibility text box over the cluster center
    plt.text(cluster_center['x'], cluster_center['y'], cluster_names[cluster_id], 
             color='white', fontsize=18, fontweight='bold', ha='center',
             bbox=dict(facecolor='red', alpha=0.4, edgecolor='white', boxstyle='round,pad=0.5'))

# 7. Final Polish & Formatting
plt.axis('off') # Removing axes for a clean, map-like look
plt.title("Nemotron's Neural Flow: How Thoughts Connect", 
          color='white', fontsize=26, pad=30, fontweight='bold')

print("✨ Visualization complete! Displaying the Neural Map...")
plt.show()

In [ ]:
#tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
trainer.model.save_pretrained(OUTPUT_DIR)
#tokenizer.save_pretrained(OUTPUT_DIR)

# 💾 Model Export & Final Archiving
### *Packaging the Trained Adapters and Notebook for Submission*

## 📖 Overview
The final stage of the pipeline involves exporting the fine-tuned LoRA adapters and the project notebook into a single, compressed archive. By using the `zip -m` command, we efficiently package the critical files—including the model weights (`safetensors`), configuration, and documentation—while clearing the working directory to optimize storage.

## 🛠️ Archiving Workflow:
1. **Model Persistence:** Saving the trained PEFT adapters to the output directory using `save_pretrained`.
2. **Notebook Relocation:** Copying the current notebook version from the input dataset to the active working directory.
3. **Smart Compression:** Bundling the four essential files into `submission.zip`.
4. **Cleanup:** Automatically removing the individual files after compression to maintain a clean workspace.

In [ ]:
import subprocess
import os

# 1. Saving the Trained Model Adapters
# This saves the 'adapter_model.safetensors' and 'adapter_config.json'
print("💾 Saving fine-tuned adapters to disk...")
trainer.model.save_pretrained(OUTPUT_DIR)

# 2. Preparing the Notebook for Packaging
input_notebook = "/kaggle/input/datasets/mah20050/my-notebook-so/nvidia-nemotron-the-wonderland-nsa.ipynb"
output_notebook = "nvidia-nemotron-the-wonderland-ns.ipynb"

# Copying the notebook to the current working directory (with space correction)
if os.path.exists(input_notebook):
    subprocess.run(f"cp {input_notebook} .", shell=True, check=True)
    print("📝 Notebook copied successfully.")

# 3. Defining the Target Files for Submission
# We include: The Notebook, Model Weights, Config, and README
target_files = f"{output_notebook} adapter_model.safetensors adapter_config.json README.md"

print("📦 Archiving files into submission.zip...")

try:
    # Using 'zip -m' to move files into the archive and delete the originals
    subprocess.run(f"zip -m submission.zip {target_files}", shell=True, check=True) 

    print("-" * 30)
    print("✅ Success! 'submission.zip' is ready and contains all 4 essential files.") 
    print("🚀 You are now ready for final submission!")

except Exception as e:
    print(f"❌ Error during archiving: {e}")
    print("💡 Tip: Ensure all 4 files exist in /kaggle/working before zipping.")